# 第 46 章：Capstone 修订闭环、公开归档与课程反馈

这个 notebook 用一个极小的 revision / archive feedback table，对比“只按修订成本选择归档候选”的 naive baseline 和“先检查共享边界、复现性、证据、学生反馈与助教反馈”的课程闭环 workflow。

In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/capstone_revision_archive_feedback_demo.csv')


def load_items(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['revision_effort_score'] = float(row['revision_effort_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_items(DATA_PATH)
print(f'Loaded {len(rows)} revision / archive items from {DATA_PATH.name}')
print('Python version:', platform.python_version())
print('Feedback source:', ordered_counts(rows, 'feedback_source'))
print('Sharing boundary:', ordered_counts(rows, 'sharing_boundary_status'))
print('Reproducibility:', ordered_counts(rows, 'reproducibility_status'))
print('Reference route:', ordered_counts(rows, 'reference_route'))


In [ ]:
reference_archive_ids = sorted(
    row['item_id']
    for row in rows
    if row['reference_route'] == 'archive_ready'
)
top4_low_effort = sorted(
    rows,
    key=lambda row: row['revision_effort_score'],
)[:4]
baseline_archive_hits = sum(
    row['reference_route'] == 'archive_ready'
    for row in top4_low_effort
)
baseline_policy_hits = sum(
    row['reference_route'] == 'policy_review_before_archive'
    for row in top4_low_effort
)
baseline_archive_precision = baseline_archive_hits / len(top4_low_effort)
baseline_policy_intrusion = baseline_policy_hits / len(top4_low_effort)

print('Reference archive-ready items:', reference_archive_ids)
print('Top-4 by low revision effort:')
for row in top4_low_effort:
    print(
        f"  {row['item_id']}: effort={row['revision_effort_score']:.2f}, "
        f"route={row['reference_route']}, area={row['package_area']}"
    )
print('Baseline archive precision:', round(baseline_archive_precision, 3))
print('Baseline policy-review intrusion:', round(baseline_policy_intrusion, 3))


In [ ]:
def route_revision_item(row):
    if row['sharing_boundary_status'] != 'clear':
        return 'policy_review_before_archive', 'sharing, license, AI-use, or student-work boundary needs review'
    if row['reproducibility_status'] != 'reproducible':
        return 'reproduce_before_archive', 'notebook, environment, or archive artifact is not reproducible yet'
    if row['evidence_status'] in {'missing', 'thin'}:
        return 'revise_before_next_run', 'course-team evidence is not strong enough for reuse'
    if row['student_feedback_status'] in {'confusing', 'blocked'}:
        return 'revise_before_next_run', 'student feedback shows the material still blocks progress'
    if row['ta_feedback_status'] in {'inconsistent', 'missing'}:
        return 'revise_before_next_run', 'TA feedback or grading calibration is not stable enough'
    return 'archive_ready', 'item is bounded, reproducible, evidence-backed, and ready to reuse'


routed_rows = []
for row in rows:
    route, reason = route_revision_item(row)
    routed = dict(row)
    routed['route'] = route
    routed['reason'] = reason
    routed_rows.append(routed)

print('Revision / archive workflow routes:')
for row in routed_rows:
    print(
        f"  {row['item_id']}: route={row['route']}, "
        f"reference={row['reference_route']}, reason={row['reason']}"
    )


In [ ]:
archive_queue = [row for row in routed_rows if row['route'] == 'archive_ready']
revise_queue = [row for row in routed_rows if row['route'] == 'revise_before_next_run']
reproduce_queue = [row for row in routed_rows if row['route'] == 'reproduce_before_archive']
policy_queue = [row for row in routed_rows if row['route'] == 'policy_review_before_archive']

workflow_accuracy = sum(
    row['route'] == row['reference_route']
    for row in routed_rows
) / len(routed_rows)
archive_precision = sum(
    row['reference_route'] == 'archive_ready'
    for row in archive_queue
) / max(len(archive_queue), 1)

print('Archive-ready queue:', [row['item_id'] for row in archive_queue])
print('Revise-before-next-run queue:', [row['item_id'] for row in revise_queue])
print('Reproduce-before-archive queue:', [row['item_id'] for row in reproduce_queue])
print('Policy-review-before-archive queue:', [row['item_id'] for row in policy_queue])
print('Workflow route accuracy:', round(workflow_accuracy, 3))
print('Structured archive precision:', round(archive_precision, 3))


In [ ]:
def revision_steps(row):
    steps = []
    if row['sharing_boundary_status'] != 'clear':
        steps.append('先复核数据、学生作品、展示录像、AI 使用或许可证边界，暂不公开。')
    if row['reproducibility_status'] != 'reproducible':
        steps.append('从干净环境复跑 notebook 或归档材料，修复路径、依赖和随机种子。')
    if row['evidence_status'] == 'missing':
        steps.append('补课后证据：失败样本、学生问题、助教记录或复现日志。')
    elif row['evidence_status'] == 'thin':
        steps.append('把零散反馈整理成至少一个可验证的修订证据。')
    if row['student_feedback_status'] in {'confusing', 'blocked'}:
        steps.append('重写学生说明，把下一步动作、输入、输出和检查标准写清楚。')
    if row['ta_feedback_status'] in {'inconsistent', 'missing'}:
        steps.append('补 TA norming case、评分锚点和无法判断时的升级路径。')
    return steps or ['可以进入归档候选；发布时仍需保留版本号和已知限制。']


for row in routed_rows:
    if row['route'] != 'archive_ready':
        print(f"\n{row['item_id']} -> {row['route']} ({row['package_area']})")
        for step in revision_steps(row):
            print(' -', step)


In [ ]:
public_archive_checklist = [
    'Data source, license, privacy boundary, and public fields are documented',
    'Notebook reruns from a clean environment with stable paths and seeds',
    'Student work, screenshots, and recordings have the required permission',
    'AI-use examples remove personal information and stale policy wording',
    'Handout, rubric, TA guide, and archive notes use the same terminology',
    'Archive package has date, version, owner, and known limitations',
]

revision_assistant_prompt = '''你是我的 capstone 课程修订与归档助手。

任务：
1. 先阅读 revision / archive feedback table，不要只按修订成本排序；
2. 先检查 sharing boundary；
3. 再检查 reproducibility、evidence、student feedback 和 TA feedback；
4. 把每个模块 route 到 archive_ready、revise_before_next_run、
   reproduce_before_archive 或 policy_review_before_archive；
5. 对每个非 archive-ready 模块输出下一轮课程修订 checklist。

输出格式：
- Archive-ready modules
- Revise-before-next-run modules
- Reproduce-before-archive modules
- Policy-review-before-archive modules
- Revision checklist by module
'''

print('Public archive checklist:')
for item in public_archive_checklist:
    print(' -', item)

print('\nAssistant prompt:')
print(revision_assistant_prompt)


In [ ]:
revision_snapshot = {
    'dataset': DATA_PATH.name,
    'n_revision_items': len(rows),
    'baseline_top4_archive_precision': round(baseline_archive_precision, 3),
    'baseline_policy_intrusion': round(baseline_policy_intrusion, 3),
    'workflow_route_accuracy': round(workflow_accuracy, 3),
    'archive_ready': [row['item_id'] for row in archive_queue],
    'revise_before_next_run': [row['item_id'] for row in revise_queue],
    'reproduce_before_archive': [row['item_id'] for row in reproduce_queue],
    'policy_review_before_archive': [row['item_id'] for row in policy_queue],
    'python_version': platform.python_version(),
}

print('Revision / archive snapshot:')
for key, value in revision_snapshot.items():
    print(f'  {key}: {value}')
